# Notebook 1 — Data Cleaning

**Goal:** Before starting the analysis, I first need to make sure the data is clean and reliable.

In this notebook, I load the three raw Inside Airbnb files, keep the columns I need, handle missing values, and save clean datasets for the rest of the project.

I keep only the columns the project actually needs. The raw listings file contains 79 columns, but most of them are not relevant to pricing, occupancy, or guest satisfaction.

**Steps in this notebook:**

1. Load the raw data
2. Clean the listings data
3. Clean the calendar data
4. Clean the reviews data
5. Save the clean files


## Step 1 — Load the raw data

The three files come directly from Inside Airbnb as compressed CSV files. Pandas can read these files without needing to unzip them first. I load each file and check its shape (the number of rows and columns) to get an overview of how much data I am working with.


In [13]:
import pandas as pd

# The raw files live in the data/raw folder, one level up from this notebook.
raw_path = "../data/raw/"

listings = pd.read_csv(raw_path + "listings.csv.gz")
calendar = pd.read_csv(raw_path + "calendar.csv.gz")
reviews = pd.read_csv(raw_path + "reviews.csv.gz")

print("Listings:", listings.shape)
print("Calendar:", calendar.shape)
print("Reviews:", reviews.shape)

Listings: (14274, 79)
Calendar: (5210011, 7)
Reviews: (635471, 6)


The shapes match what I expected. The listings dataset has one row per rental, the calendar dataset has one row per listing per day, and the reviews dataset has one row per guest review.


## Step 2 — Clean the listings data

The listings dataset is the main dataset for this project. I start by keeping only the columns I need for the analysis. This keeps the dataset easier to manage and makes the analysis easier to explain.

Some column names use abbreviations, so I briefly explain them below:

* `number_of_reviews_ltm` — "ltm" stands for "last twelve months", so this column shows the number of reviews a listing received during the past year.

* `estimated_occupancy_l365d` — "l365d" stands for "last 365 days". This column contains Inside Airbnb's estimate of how many nights a listing was booked during the past year.


In [14]:
columns_to_keep = [
    "id", "name", "host_id", "host_is_superhost",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "price", "minimum_nights", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value",
    "reviews_per_month", "instant_bookable",
]

listings = listings[columns_to_keep]
print("Listings now has", listings.shape[1], "columns")

Listings now has 26 columns


### Clean the price column

The price column is stored as text, for example `$105.00`, so I need to convert it into a numeric column before I can perform any calculations. To do this, I first make sure the values are treated as text, remove the dollar sign and comma, and then convert the result to a number.

I use `errors="coerce"` so that any unexpected values are turned into missing values (`NaN`) instead of causing an error. This keeps the cleaning process running smoothly and makes it easier to identify problematic values later.


In [15]:
listings["price"] = listings["price"].astype(str)
listings["price"] = listings["price"].str.replace("$", "", regex=False)
listings["price"] = listings["price"].str.replace(",", "", regex=False)
listings["price"] = pd.to_numeric(listings["price"], errors="coerce")

print("Price column type is now:", listings["price"].dtype)
print(listings["price"].describe().round(1))

Price column type is now: float64
count     9264.0
mean       201.2
std       1657.0
min          5.0
25%         70.0
50%        104.0
75%        160.0
max      50000.0
Name: price, dtype: float64


### Check missing values

I count how many values are missing in each column. This helps me see which columns can be used as they are and which ones need further cleaning or special handling.


In [16]:
missing = listings.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values by column:")
print(missing)

Missing values by column:
estimated_revenue_l365d      5010
price                        5010
beds                         4995
review_scores_value          3318
review_scores_location       3317
review_scores_rating         3314
review_scores_cleanliness    3314
reviews_per_month            3314
bedrooms                     2024
host_is_superhost             164
dtype: int64


There are two main types of missing values, and I handle them differently on purpose:

* **Price** is missing for many listings. These listings currently do not have a nightly price, so rather than guessing values, I keep them and create a flag that allows me to include or exclude them later if needed.

* **Review scores** are missing for listings that have not received any reviews yet. This is expected for new listings, so I leave these values as missing rather than filling them with an artificial score, which could distort the averages.


In [17]:
# Flag listings that have a price.
# This allows priced and unpriced listings to be analysed separately.
# without removing any rows.
listings["has_price"] = listings["price"].notna()

print("Listings with a price:", listings["has_price"].sum())
print("Listings without a price:", (~listings["has_price"]).sum())

Listings with a price: 9264
Listings without a price: 5010


### Tidy the superhost column

The `host_is_superhost` column uses "t" and "f" to represent true and false. I convert these values to the clearer labels "Yes" and "No" to make them easier to read in charts and in Tableau.

A small number of listings have missing values in this column. Instead of removing these listings, I label them as "Unknown" so the total number of listings stays accurate.


In [18]:
listings["host_is_superhost"] = listings["host_is_superhost"].map(
    {"t": "Yes", "f": "No"}
)
listings["host_is_superhost"] = listings["host_is_superhost"].fillna("Unknown")

print(listings["host_is_superhost"].value_counts())

host_is_superhost
No         10578
Yes         3532
Unknown      164
Name: count, dtype: int64


The `instant_bookable` column uses the same "t" and "f" values, so I convert these values in the same way.


In [19]:
listings["instant_bookable"] = listings["instant_bookable"].map(
    {"t": "Yes", "f": "No"}
)
print(listings["instant_bookable"].value_counts())

instant_bookable
No     9908
Yes    4366
Name: count, dtype: int64


### Create an occupancy rate column

The `estimated_occupancy_l365d` column shows the estimated number of nights a listing was booked during the last 365 days. I convert this value into a percentage because it is easier to understand and compare than the raw number of booked nights.

For example, 91 booked nights corresponds to an occupancy rate of about 25 percent, meaning the listing was booked for roughly a quarter of the year.


In [20]:
listings["occupancy_rate"] = (listings["estimated_occupancy_l365d"] / 365) * 100
listings["occupancy_rate"] = listings["occupancy_rate"].round(1)

print(listings["occupancy_rate"].describe().round(1))

count    14274.0
mean        21.2
std         27.8
min          0.0
25%          0.0
50%          3.6
75%         44.9
max         69.9
Name: occupancy_rate, dtype: float64


### Check for duplicate listings

Each listing should appear only once, so I check for duplicate listing IDs. A result of zero means that every listing is unique.


In [21]:
duplicate_count = listings["id"].duplicated().sum()
print("Duplicate listing ids:", duplicate_count)

Duplicate listing ids: 0


## Step 3 — Clean the calendar data

The calendar dataset contains information about whether each listing is available on a given date. Two columns need cleaning: the `available` column uses "t" and "f", and the `price` column is stored as text, just like in the listings dataset.

I also convert the `date` column from text to a proper date format so it can be used for time-based analysis later in the project.


In [22]:
# Convert availability values from "t"/"f" to True/False.
calendar["is_available"] = calendar["available"].map({"t": True, "f": False})

# Clean the price column the same way as before. Most calendar rows have no
# price, so I force the column to text first to keep the steps safe.
calendar["price"] = calendar["price"].astype(str)
calendar["price"] = calendar["price"].str.replace("$", "", regex=False)
calendar["price"] = calendar["price"].str.replace(",", "", regex=False)
calendar["price"] = pd.to_numeric(calendar["price"], errors="coerce")

# Convert the date column to a proper date format.
calendar["date"] = pd.to_datetime(calendar["date"])

print(calendar[["listing_id", "date", "is_available", "price"]].head())

   listing_id       date  is_available  price
0     1358910 2025-09-24         False    NaN
1     1358910 2025-09-25         False    NaN
2     1358910 2025-09-26          True    NaN
3     1358910 2025-09-27          True    NaN
4     1358910 2025-09-28          True    NaN


## Step 4 — Clean the reviews data

The reviews dataset is mainly used to track how many reviews each listing receives over time. I convert the `date` column to a proper date format, remove any reviews with missing dates, and create a `review_year` column so I can analyze recent activity later in the project.


In [23]:
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")

# Remove reviews with no date, because they cannot be used in any analysis
# that depends on when the review was written.
rows_before = len(reviews)
reviews = reviews.dropna(subset=["date"])
print("Dropped", rows_before - len(reviews), "reviews with no date")

# Pull the year out of each review date into its own column.
reviews["review_year"] = reviews["date"].dt.year
print(reviews["review_year"].value_counts().sort_index().tail(6))

Dropped 0 reviews with no date
review_year
2020     28608
2021     39305
2022     77312
2023     95640
2024    121499
2025    101825
Name: count, dtype: int64


## Step 5 — Save the clean files

I save the three cleaned files in the `data/clean` folder. The rest of the project uses these cleaned files instead of the raw ones, so the cleaning process only needs to be done once.


In [ ]:
clean_path = "../data/clean/"

listings.to_csv(clean_path + "listings_clean.csv", index=False)
calendar.to_csv(clean_path + "calendar_clean.csv", index=False)
reviews.to_csv(clean_path + "reviews_clean.csv", index=False)

print("Saved clean files:")
print("- listings_clean.csv :", listings.shape)
print("- calendar_clean.csv :", calendar.shape)
print("- reviews_clean.csv  :", reviews.shape)

## Insight summary

The data is now clean and ready for analysis:

* **Listings:** 14,274 rows. The price column is now numeric. About 9,300 listings have a nightly price, while the remaining listings currently do not have one. Rather than guessing values, I flagged these listings so they can be included or excluded when needed.

* **Calendar:** availability and price have been cleaned, and the date column has been converted to a proper date format.

* **Reviews:** dates have been cleaned, and I created a `review_year` column for time-based analysis.

## Business recommendation

For pricing analysis, I will use only listings with a valid price and remove extreme values in the next notebook to prevent averages from being distorted.

Occupancy analysis can include all listings, and missing review scores will remain as missing values because they simply indicate that a listing has not received any reviews yet.
